# Repeat-purchase prediction model

**Goal:** for every customer who bought before a cutoff date, predict whether they will buy again in the next 90 days.

**Why this setup (and not row-level):** the data is one row per product line, but the business question is about customers. So we aggregate to one row per customer (RFM-style features) using only data *before* the cutoff, and build the label from data *after* the cutoff. This avoids data leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

## 1. Load data

We load `sales_df.csv` (already filtered: Quantity > 0, Price > 0, Customer ID not null).
We do **not** use the outlier-removed `df` from EDA3 here: dropping rows would change each customer's true order history and distort the features. Big orders are real behaviour, and tree models handle them well.

In [ ]:
df = pd.read_csv('../data/sales_df.csv', index_col=0)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Sales'] = df['Quantity'] * df['Price']
df['Customer ID'] = df['Customer ID'].astype(int)

print(df.shape)
print(df['InvoiceDate'].min(), '->', df['InvoiceDate'].max())

## 2. Define cutoff and prediction window

Last 90 days of the data are the **future window** (used only for the label). Everything before is the **history window** (used for features).

In [ ]:
PREDICTION_DAYS = 90

end_date = df['InvoiceDate'].max()
cutoff = (end_date - pd.Timedelta(days=PREDICTION_DAYS)).normalize()
print('cutoff:', cutoff, '| end of data:', end_date)

history = df[df['InvoiceDate'] < cutoff].copy()
future  = df[df['InvoiceDate'] >= cutoff].copy()

print('history rows:', len(history), '| future rows:', len(future))

## 3. Build features (one row per customer, history only)

In [ ]:
def build_features(history, cutoff):
    g = history.groupby('Customer ID')

    # invoice-level table (one row per invoice) for order-based features
    inv = (history.groupby(['Customer ID', 'Invoice'])
                  .agg(InvoiceDate=('InvoiceDate', 'min'),
                       order_value=('Sales', 'sum'),
                       order_items=('Quantity', 'sum'))
                  .reset_index()
                  .sort_values(['Customer ID', 'InvoiceDate']))

    feats = pd.DataFrame(index=g.size().index)
    feats.index.name = 'Customer ID'

    # --- RFM ---
    last_purchase = g['InvoiceDate'].max()
    first_purchase = g['InvoiceDate'].min()
    feats['recency_days'] = (cutoff - last_purchase).dt.days
    feats['tenure_days']  = (cutoff - first_purchase).dt.days
    feats['frequency']    = inv.groupby('Customer ID')['Invoice'].nunique()
    feats['monetary']     = g['Sales'].sum()

    # --- order behaviour ---
    feats['avg_order_value']  = inv.groupby('Customer ID')['order_value'].mean()
    feats['max_order_value']  = inv.groupby('Customer ID')['order_value'].max()
    feats['avg_items_per_order'] = inv.groupby('Customer ID')['order_items'].mean()
    feats['n_unique_products'] = g['StockCode'].nunique()
    feats['avg_price'] = g['Price'].mean()

    # --- gap between orders ---
    gaps = (inv.groupby('Customer ID')['InvoiceDate']
               .apply(lambda s: s.diff().dt.days.dropna()))
    gap_mean = gaps.groupby(level=0).mean()
    feats['avg_days_between_orders'] = gap_mean
    # customers with a single order have no gap -> fill with tenure (large = "slow")
    feats['avg_days_between_orders'] = feats['avg_days_between_orders'].fillna(feats['tenure_days'])

    # --- recent activity windows ---
    for d in (30, 90, 180):
        recent = history[history['InvoiceDate'] >= cutoff - pd.Timedelta(days=d)]
        feats[f'orders_last_{d}d'] = recent.groupby('Customer ID')['Invoice'].nunique()
        feats[f'spend_last_{d}d']  = recent.groupby('Customer ID')['Sales'].sum()

    # --- country ---
    country = g['Country'].agg(lambda s: s.mode().iloc[0])
    feats['is_uk'] = (country == 'United Kingdom').astype(int)

    return feats.fillna(0)

X_all = build_features(history, cutoff)
X_all.head()

## 4. Build the target

`repeat_purchase = 1` if the customer placed at least one order in the next 90 days.

In [ ]:
buyers_future = set(future['Customer ID'].unique())
y_all = pd.Series(X_all.index.isin(buyers_future).astype(int),
                  index=X_all.index, name='repeat_purchase')

print('customers:', len(X_all))
print(y_all.value_counts())
print(y_all.value_counts(normalize=True).round(3))

## 5. Train / test split

Split by customer (each customer is one row, so no customer is in both sets). Stratified to keep the class ratio equal.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)

print(X_train.shape, X_test.shape)

## 6. Models

Start with a simple baseline, then try tree models. Skewed features (money, counts) are fine for trees; for logistic regression we apply `log1p` first.

In [ ]:
skewed = ['monetary', 'avg_order_value', 'max_order_value', 'avg_items_per_order',
          'frequency', 'n_unique_products', 'spend_last_30d', 'spend_last_90d', 'spend_last_180d']

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

# built-in np.log1p (not a custom function) so the saved model loads anywhere, e.g. in FastAPI
logistic_prep = ColumnTransformer(
    [('log', FunctionTransformer(np.log1p), skewed)],
    remainder='passthrough')

models = {
    'LogisticRegression': Pipeline([
        ('prep', logistic_prep),
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'RandomForest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:22s} CV ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})')

## 7. Evaluate on the test set

In [ ]:
rows = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({'model': name,
                 'ROC-AUC': roc_auc_score(y_test, proba),
                 'PR-AUC': average_precision_score(y_test, proba)})

results = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results

In [ ]:
best_name = results.loc[0, 'model']
best_model = fitted[best_name]
print('Best model:', best_name)

proba = best_model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(classification_report(y_test, pred, digits=3))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[0])
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=ax[1])
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', ax=ax[2])
ax[2].set_xlabel('predicted'); ax[2].set_ylabel('actual')
plt.tight_layout(); plt.show()

## 8. Pick a decision threshold

0.5 is not always the best cut-off. Choose it based on the business use (e.g. a retention campaign: do you prefer catching more repeat buyers, or contacting fewer people?).

In [ ]:
thresholds = np.arange(0.2, 0.81, 0.05)
tbl = []
for t in thresholds:
    p = (proba >= t).astype(int)
    tp = ((p == 1) & (y_test == 1)).sum()
    fp = ((p == 1) & (y_test == 0)).sum()
    fn = ((p == 0) & (y_test == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    tbl.append({'threshold': round(t, 2), 'precision': precision, 'recall': recall, 'f1': f1})
pd.DataFrame(tbl).round(3)

## 9. Feature importance (permutation, on the test set)

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, scoring='roc_auc',
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
imp.plot(kind='barh', figsize=(7, 6), title='Permutation importance (drop in ROC-AUC)')
plt.tight_layout(); plt.show()

## 10. Save the model

The saved pipeline + the feature list is what the FastAPI service will load later.

In [ ]:
import os, json
os.makedirs('../models', exist_ok=True)

joblib.dump(best_model, '../models/repeat_purchase_model.joblib')
with open('../models/feature_columns.json', 'w') as f:
    json.dump(list(X_all.columns), f)

print('saved model:', best_name)

## 11. Sanity check: reload and predict

In [ ]:
loaded = joblib.load('../models/repeat_purchase_model.joblib')
sample = X_test.head(5)
print(pd.DataFrame({'customer': sample.index,
                    'prob_repeat': loaded.predict_proba(sample)[:, 1].round(3),
                    'actual': y_test.loc[sample.index].values}))